# 09 — Consolidación y submission

**Objetivo.** Recoger el mejor enfoque de cada índice, verificar shapes y fechas, y producir el bloque 252×6 listo para pegar en el template Excel del profesor.

**Qué hace.** Cargar 6 JSON → tabla comparativa → generar predicciones → validar (shape + NaN/Inf + fechas vs template) → copiar al portapapeles para pegar en B2:G253 del template.

**Qué NO hace.** No entrena nada. No modifica los JSON ni los modelos.

**Inputs.** `results/index_A.json` … `results/index_F.json`, CSV de `data/`, modelos en `models/`, template en `data/`.

**Output principal.** Portapapeles con 252×6 valores (sin cabecera, sin índice) → pegar en B2:G253 del template.

---

### Convención de ancla (CRÍTICO — una sola fuente de verdad)

El precio inicial para reconstruir precios desde log-retornos es **siempre** `train_indices[col].iloc[-1]` — el último precio real conocido de train. **Nunca** se usa un precio del período de test como ancla.

### Template de entrega

Fichero: `data/hackaton_miax_prediccion_prod_rmse_20260530_v1_submission_template.xlsx`, hoja `submission`.  
Columnas: `Date | pred_index_a | pred_index_b | pred_index_c | pred_index_d | pred_index_e | pred_index_f`  
Fechas ya rellenas en columna A: **2028-12-13 → 2029-08-21** (252 días).  
A rellenar: **celdas B2:G253** — 252 filas × 6 columnas de predicciones numéricas.

## 0. GPU workaround + imports + constantes

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from utils import (
    load_hackathon_data,
    predict_autoregressive, precios_a_logret,
    align_aux_features,
    baseline_flat, baseline_drift, baseline_random_walk,
    plot_rmse_by_index,
    DATA_DIR, N_DAYS_PRED, V_IN_SHARED, INDEX_COLS
)

# ── Ruta del template de entrega (no modificar)
TEMPLATE_PATH = 'data/hackaton_miax_prediccion_prod_rmse_20260530_v1_submission_template.xlsx'
TEMPLATE_SHEET = 'submission'

# ── Mapeo nombres internos → nombres del template
# Aunque el método principal pega sin cabecera, este mapeo se usa en exportaciones
# con cabecera y es la única fuente de verdad para el rename si el template cambia.
COL_MAP = {
    'Index_A': 'pred_index_a',
    'Index_B': 'pred_index_b',
    'Index_C': 'pred_index_c',
    'Index_D': 'pred_index_d',
    'Index_E': 'pred_index_e',
    'Index_F': 'pred_index_f',
}
OUTPUT_COLS = list(COL_MAP.keys())   # orden fijo: A, B, C, D, E, F

## 1. Cargar los 6 JSON de results/

Si falta alguno, el proceso se detiene con un mensaje claro.

In [ ]:
INDEX_LABELS = ['A', 'B', 'C', 'D', 'E', 'F']
infos = {}

for lbl in INDEX_LABELS:
    path = f'results/index_{lbl}.json'
    if not os.path.exists(path):
        raise FileNotFoundError(
            f'Falta {path} — ejecutar primero el notebook 0{3 + INDEX_LABELS.index(lbl)}_index_{lbl}.ipynb'
        )
    with open(path) as f:
        infos[f'Index_{lbl}'] = json.load(f)

print(f'[OK] {len(infos)} JSONs cargados')

## 2. Tabla comparativa de RMSE por índice

In [ ]:
rows = [
    {'index': col, 'approach': info['approach_type'], 'strategy': info['strategy'],
     'owner': info['owner'], 'rmse_backtest': info['rmse_backtest_252d']}
    for col, info in infos.items()
]
tabla = pd.DataFrame(rows).set_index('index').sort_values('rmse_backtest')
print(tabla.to_string())
print(f'\nRMSE medio estimado: {tabla["rmse_backtest"].mean():,.0f}')

plot_rmse_by_index(
    {r['index']: r['rmse_backtest'] for r in rows},
    title='RMSE backtest 252d por índice — enfoque elegido'
)

## 3. Carga de datos

In [ ]:
data = load_hackathon_data(DATA_DIR)
idx  = data['train_indices']

## 4. Generar predicciones para los 252 días de producción

Switch sobre `approach_type`. Ancla = `train_indices[col].iloc[-1]` para todos los índices.

## ⚠️ GAP DE DATOS AUXILIARES — DECISIÓN PENDIENTE ANTES DE EJECUTAR

### Macro (Index_C): `test_macro_factors.csv` tiene **173 filas** para un horizonte de **252 días**

- Los 252 días del rollout son **calendario completo** (incluye fines de semana y festivos).
- `test_macro_factors.csv` solo tiene los **días hábiles** (≈173), igual que `train_macro_factors.csv`.
- Faltan **~79 días** de datos macro para completar el horizonte.

**Decisión que hay que tomar (el código implementa ffill por defecto):**
| Opción | Descripción | Cuándo usarla |
|--------|-------------|---------------|
| `ffill` (default) | Repite el último valor conocido | Si la macro cambia lentamente (tipos, oro) |
| `bfill` | Rellena con el valor siguiente disponible | Si queremos "anticipar" el siguiente dato |
| `zero` | Rellena con cero | Si las features están normalizadas y 0 es neutro |
| `mean` | Rellena con la media histórica | Si no hay señal específica del período gap |

**El código imprimirá cuántos días se rellenaron.** Si ves `⚠️ faltan X días → rellenando con ffill`, revisa si la opción elegida es correcta antes de entregar.

### Network (Index_F): `test_network_metrics.csv` tiene **252 filas** → sin gap.

> Ver también: **CLAUDE.md → § Deuda técnica, punto 2** para el contexto completo y las opciones de relleno.

In [ ]:
import keras

predicciones    = {}   # col → array(252) de los índices que funcionaron
indices_fallidos = {}  # col → mensaje de error

for col, info in infos.items():
    try:
        serie          = idx[col].dropna().values
        at             = info['approach_type']
        precio_inicial = float(idx[col].iloc[-1])   # ancla única: último precio real de train

        # ── Rama NN / NN-ensemble ─────────────────────────────────────────────
        if at in ('nn', 'nn_ensemble'):
            model      = keras.models.load_model(info['model_path'])
            predict_fn = lambda x, m=model: m.predict(x, verbose=0).ravel()[0]
            v_in       = info['v_in']
            n_features = info.get('n_features', 1)   # retrocompat: JSONs sin el campo → 1

            # Ventana inicial: (v_in, n_features) — NO hardcodeado a 1 canal
            log_rets = precios_a_logret(serie).astype('float32')
            ventana  = np.zeros((v_in, n_features), dtype='float32')
            ventana[:, 0] = log_rets[-v_in:]          # canal 0: log-rets del precio

            aux_data_prod = None
            if n_features > 1:
                aux_test_src = info['aux_test_source']   # 'test_macro' | 'test_network'
                aux_cols     = info['aux_columns']        # lista ordenada de columnas aux
                aux_src      = info['aux_source']         # 'train_macro' | 'train_network'

                if aux_test_src not in data:
                    raise KeyError(
                        f"aux_test_source='{aux_test_src}' no cargado en data. "
                        f"Verificar que el CSV existe en {DATA_DIR}"
                    )

                # ── Ventana inicial: últimos v_in valores aux de train ────────
                # align_aux_features devuelve DF indexado igual que idx
                if aux_src in data:
                    train_aux = align_aux_features(idx, data[aux_src], aux_cols)
                    ventana[:, 1:] = train_aux.values[-v_in:].astype('float32')
                else:
                    print(f'  [AVISO] {col}: aux_source={aux_src} no cargado — '
                          f'ventana aux inicializada a cero')

                # ── aux_data para el rollout productivo: (N_DAYS_PRED, k) ────
                test_aux_arr  = data[aux_test_src][aux_cols].values.astype('float32')
                n_disponibles = len(test_aux_arr)

                if n_disponibles < N_DAYS_PRED:
                    n_relleno = N_DAYS_PRED - n_disponibles
                    print(f'  ⚠️  {col}: {aux_test_src} tiene {n_disponibles} filas, '
                          f'faltan {n_relleno} días → rellenando con ffill '
                          f'(ver celda de aviso arriba para alternativas)')
                    last_row    = test_aux_arr[-1:]
                    relleno     = np.repeat(last_row, n_relleno, axis=0)
                    test_aux_arr = np.vstack([test_aux_arr, relleno])
                elif n_disponibles > N_DAYS_PRED:
                    test_aux_arr = test_aux_arr[:N_DAYS_PRED]

                aux_data_prod = test_aux_arr   # (N_DAYS_PRED, k)

            preds = predict_autoregressive(
                predict_fn, ventana, N_DAYS_PRED,
                precio_inicial=precio_inicial,
                aux_data=aux_data_prod
            )

        # ── Ghost ─────────────────────────────────────────────────────────────
        elif at == 'ghost':
            src          = info['ghost_source_index']
            lag          = info['ghost_lag']
            fuente_train = idx[src].dropna().values

            preds_ghost = list(fuente_train[-lag:]) if lag > 0 else []
            n_rest = N_DAYS_PRED - lag
            if n_rest > 0:
                preds_ghost.extend(baseline_drift(fuente_train, n_rest))
            preds = np.array(preds_ghost[:N_DAYS_PRED])

        # ── Baselines ─────────────────────────────────────────────────────────
        elif at == 'baseline_flat':
            preds = baseline_flat(serie, N_DAYS_PRED)
        elif at == 'baseline_drift':
            preds = baseline_drift(serie, N_DAYS_PRED)
        elif at == 'baseline_rw':
            preds = baseline_random_walk(serie, N_DAYS_PRED)
        else:
            raise ValueError(f'approach_type desconocido: {at!r}')

        predicciones[col] = preds
        print(f'  [OK]    {col}: {len(preds)} días  '
              f'min={preds.min():.0f}  max={preds.max():.0f}')

    except Exception as exc:
        indices_fallidos[col] = str(exc)
        print(f'  [FALLO] {col}: {exc}')

# ── Resumen ────────────────────────────────────────────────────────────────────
print(f'\n{"="*60}')
print(f'OK    : {sorted(predicciones.keys())}')
if indices_fallidos:
    print(f'FALLOS: {sorted(indices_fallidos.keys())}')
    for col_f, err in indices_fallidos.items():
        print(f'  {col_f}: {err}')
    print('\n[AVISO] Completar los índices fallidos antes de entregar.')
else:
    print('Todos los índices generados correctamente.')

# ── DataFrame de predicciones (solo los disponibles) ──────────────────────────
available_cols = [c for c in OUTPUT_COLS if c in predicciones]
df_pred = pd.DataFrame({c: predicciones[c] for c in available_cols})

## 5. Validación: shape, NaN/Inf y fechas vs template

**Las fechas son críticas.** Si las 252 fechas de `test_dates.csv` no coinciden exactamente con las del template (2028-12-13 → 2029-08-21), estamos pegando los valores en el día equivocado — un fallo silencioso que quemaría un intento.

In [ ]:
# ── 1. Shape ──────────────────────────────────────────────────────────────────
assert df_pred.shape == (N_DAYS_PRED, 6), \
    f'Shape incorrecto: {df_pred.shape}  (esperado: ({N_DAYS_PRED}, 6))'

# ── 2. NaN / Inf ──────────────────────────────────────────────────────────────
nan_count = int(df_pred.isnull().sum().sum())
inf_count = int(np.isinf(df_pred.values).sum())
assert nan_count == 0, f'NaN detectados: {nan_count}'
assert inf_count == 0, f'Inf detectados: {inf_count}'

print(f'[OK] shape={df_pred.shape}  NaN={nan_count}  Inf={inf_count}')

# ── 3. Fechas: cargar template y comparar con test_dates.csv ──────────────────
template_df    = pd.read_excel(TEMPLATE_PATH, sheet_name=TEMPLATE_SHEET)
template_dates = pd.to_datetime(template_df['Date'])

print(f'Template: {len(template_dates)} fechas  '
      f'({template_dates.iloc[0].date()} → {template_dates.iloc[-1].date()})')

if 'test_dates' in data:
    td = data['test_dates']
    # load_hackathon_data usa index_col=0, parse_dates=True → fechas en el índice
    if isinstance(td.index, pd.DatetimeIndex):
        test_dates_arr = td.index
    else:
        test_dates_arr = pd.to_datetime(td.iloc[:, 0])

    mismatches = (pd.DatetimeIndex(test_dates_arr) != pd.DatetimeIndex(template_dates.values)).sum()
    if mismatches > 0:
        raise ValueError(
            f'FECHAS NO COINCIDEN: {mismatches} diferencias entre test_dates.csv y el template. '
            f'Revisar antes de pegar — pegar en el día equivocado quema un intento.'
        )
    print(f'[OK] test_dates.csv coincide EXACTAMENTE con el template ({len(test_dates_arr)} fechas)')
else:
    print('[AVISO] test_dates.csv no cargado — validación de fechas omitida.')
    print('        Referencia manual: el template va de 2028-12-13 a 2029-08-21.')

## 6. Salida principal — pegar en B2:G253 del template

`to_clipboard(index=False, header=False)` produce **252 filas × 6 columnas**, separadas por tabulador, sin cabecera ni índice — el formato exacto para pegar en Excel.

**Instrucciones:**
1. Ejecutar la celda siguiente.
2. Abrir `data/hackaton_miax_prediccion_prod_rmse_20260530_v1_submission_template.xlsx`, hoja `submission`.
3. Seleccionar la celda **B2**.
4. **Ctrl+V** — los valores rellenan B2:G253 automáticamente.
5. Verificar visualmente que la columna B (pred_index_a) corresponde a Index_A, etc.

In [ ]:
# Bloque para pegar: 252 filas × 6 cols, sin cabecera, sin índice
# Orden de columnas: A, B, C, D, E, F — coincide con el orden de columnas del template
df_pred[OUTPUT_COLS].to_clipboard(index=False, header=False)

print('[PORTAPAPELES] 252×6 valores copiados.')
print('  → Abrir el template, seleccionar B2, Ctrl+V.')
print(f'  Columnas pegadas en orden: {list(COL_MAP.values())}')
print(f'  Rango de fechas del template: 2028-12-13 → 2029-08-21')

## 7. Respaldo — texto para copiar a mano

Si el portapapeles no funciona en la máquina (p.ej. en un entorno remoto), copiar el bloque impreso y pegarlo a mano. El separador es coma; al pegar en Excel usar "Datos → Texto en columnas" si no se divide automáticamente.

In [ ]:
# Mismo bloque, formato CSV (sin cabecera, sin índice) — para copiar a mano si hace falta
print('=== VALORES (sin cabecera, sin índice — copiar todo el bloque) ===')
print(df_pred[OUTPUT_COLS].to_csv(index=False, header=False))

## 8. [OPCIONAL] Escribir el template .xlsx directamente con openpyxl

Segunda vía de entrega si los métodos anteriores fallan. Carga el template, escribe B2:G253 y guarda una copia (`results/submission_filled.xlsx`). No sobrescribe el template original.

Descomentar y ejecutar si se necesita.

In [ ]:
# import openpyxl
#
# wb = openpyxl.load_workbook(TEMPLATE_PATH)
# ws = wb[TEMPLATE_SHEET]
#
# # B2:G253 → columna 2 (B) a 7 (G), filas 2 a 253
# for j, col in enumerate(OUTPUT_COLS):
#     for i, val in enumerate(df_pred[col].values):
#         ws.cell(row=i + 2, column=j + 2, value=float(val))
#
# os.makedirs('results', exist_ok=True)
# output_xlsx = 'results/submission_filled.xlsx'
# wb.save(output_xlsx)
# print(f'[OK] Template relleno guardado en: {output_xlsx}')
# print('     Abrir y verificar B2:G253 antes de entregar.')